# Versión comentada del taller — notebook guía 3

## `3-chatbot-con-mcp-y-gradio.ipynb`: chatbot con MCP, Ollama y Gradio

Esta es una copia del notebook guía del profesor, presentado en la sesión 6 del curso (Model Context Protocol), donde la lógica del código original se conserva sin modificaciones; únicamente se han añadido comentarios de documentación y celdas de explicación paso a paso.

El objetivo de esta versión es que pueda entenderse de forma autónoma, incluso por una persona sin experiencia en programación. Por ello, cada línea de código relevante está comentada y, antes de cada bloque, se explica qué hace, por qué se hace y qué resultado produce.

El notebook original, sin comentarios, se encuentra en este mismo repositorio bajo el nombre `3-chatbot-con-mcp-y-gradio.ipynb`. En el notebook `proyecto_mcp_chatbot_recetas.ipynb` se encuentra la solución propia del taller, con servidores MCP nuevos (documentos de recetas y utilidades de cocina) en lugar de la calculadora y el clima de los ejemplos.


# Chatbot con MCP, Ollama y Gradio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/3-chatbot-con-mcp-y-gradio.ipynb)

Este notebook es un ejemplo avanzado y opcional de la sesión. No introduce conceptos nuevos: simplemente toma el agente con herramientas MCP del notebook anterior y le monta encima una interfaz conversacional con Gradio, siguiendo el estilo visto antes en la sesión de RAG con LangChain. La idea es cerrar la unidad mostrando cómo pasar de una integración técnica a una experiencia de usuario lista para demostración.

### Referencias
- [Gradio](https://www.gradio.app/)
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Ollama](https://ollama.com/)


## Celda 1 — Preparación del entorno

Se ocultan los avisos de advertencia de Python y se pregunta al sistema si el notebook está ejecutándose dentro de Google Colab (la plataforma gratuita en la nube). La respuesta queda guardada en la variable `IN_COLAB` (`True` si se está en Colab, `False` si se corre en un computador propio), porque más adelante se usará para decidir qué pasos de instalación y de arranque de servidores conviene hacer.

In [1]:
# --- Preparación inicial del entorno ---

# 'import warnings' carga el módulo de avisos de Python. Este módulo emite
# mensajes de advertencia (no errores) cuando algo no está del todo bien,
# por ejemplo cuando una librería está por quedar obsoleta.
import warnings

# Aquí se pide suprimir esos avisos para que la salida del notebook quede limpia.
warnings.filterwarnings('ignore')

# --- ¿Estamos ejecutándonos en Google Colab? ---
# El bloque 'try/except' intenta ejecutar un trozo de código y, si falla
# con el error previsto (aquí 'ImportError'), ejecuta el bloque 'except'
# en lugar de detener el programa.
try:
    # El módulo 'google.colab' solo existe dentro de Colab. Si este import
    # funciona, es señal clara de que se está corriendo en Colab.
    import google.colab
    IN_COLAB = True
except ImportError:
    # Si la importación falla, no se está en Colab sino en un computador
    # propio (por ejemplo, con VS Code).
    IN_COLAB = False

## Celda 2 — Descarga de los archivos auxiliares (solo en Colab)

El agente de este notebook depende de dos archivos Python ajenos al notebook: los servidores MCP de la calculadora y del clima. Cuando Colab abre el notebook directamente desde GitHub, trae el notebook pero no siempre los demás archivos del repositorio. Esta celda solo actúa en Colab: crea la carpeta `mcp_servers` y descarga allí los tres archivos necesarios desde el repositorio del curso. En un computador propio no descarga nada, porque los archivos ya están en la carpeta.

In [2]:
# En Colab, cuando se abre el notebook desde GitHub, los archivos auxiliares del repo
# no siempre se descargan. Esta celda garantiza que mcp_servers exista.

# Se lee la variable IN_COLAB de la celda anterior de forma segura: si por
# alguna razón no existe, se asume False ('globals()' es el diccionario de
# todas las variables creadas hasta ahora en el notebook).
is_colab = bool(globals().get('IN_COLAB', False))
if is_colab:
    # 'mkdir -p' crea la carpeta mcp_servers (si ya existía, no pasa nada).
    !mkdir -p Sesion6/mcp_servers
    # 'wget -q -O destino url' descarga un archivo de internet de forma silenciosa
    # (-q) y lo guarda con el nombre indicado (-O). Se bajan el inicializador del
    # paquete y los dos servidores MCP del curso.
    !wget -q -O Sesion6/mcp_servers/__init__.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/__init__.py
    !wget -q -O Sesion6/mcp_servers/calculator_mcp_server.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/calculator_mcp_server.py
    !wget -q -O Sesion6/mcp_servers/weather_mcp_server.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/weather_mcp_server.py
    print('MCP servers descargados para Colab.')
else:
    print('Entorno local detectado: no se descargan archivos auxiliares.')

Entorno local detectado: no se descargan archivos auxiliares.


## Celda 3 — Instalación de las librerías (solo en Colab)

Esta celda instala, de una sola vez, todas las librerías que el notebook necesita. Solo se ejecuta en Colab (el `test ... && pip install` es la forma de decir «instala si estamos en Colab»); en local se asume que las librerías ya están instaladas (p. ej. con `pip install -r requirements.txt`).

Las más importantes: `mcp` (el SDK oficial del protocolo Model Context Protocol, con el que se escriben los servidores), `langchain-mcp-adapters` (el puente entre servidores MCP y LangChain), `langchain`, `langchain-core` y `langgraph` (el armazón del agente que decide cuándo usar las herramientas), `langchain-ollama` (el conector con el modelo local Ollama), `httpx` (peticiones HTTP), `gradio` (la interfaz de chat) y `ollama` `colab-xterm` (utilidades del entorno).
Las versiones están acotadas (`>=1.24,<2.0`, etc.) por si la librería cambia a futuro: así el cuaderno sigue funcionando igual.

In [3]:
# Solo en Colab: instala el SDK de MCP, los adaptadores de LangChain, el
# armazon del agente, el conector de Ollama, httpx, Gradio y utilidades.
!test '{IN_COLAB}' = 'True' && pip install "mcp>=1.24.0,<2.0.0" "langchain-mcp-adapters>=0.3.0,<0.4.0" langchain langchain-core langchain-ollama langgraph httpx gradio ollama colab-xterm

# En local, instala/actualiza con: pip install -r requirements.txt

### Cargando a Ollama

Usaremos el mismo modelo local y la misma idea de tools de los notebooks previos para que lo único nuevo aquí sea la interfaz.


## Celda 5 — Instalación de Ollama (solo en Colab)

Ollama es el programa que sirve modelos de lenguaje en el propio computador. La celda instala su componente `zstd` (compresión que usa el instalador) y luego ejecuta el instalador oficial **solo si Ollama no está ya instalado**: `type ollama` busca el programa y, si existe, imprime el mensaje «ya está instalado».

In [ ]:
# 'zstd' es una herramienta de compresion que necesita el instalador de Ollama.
!sudo apt install zstd -y
# Instala Ollama solo si 'ollama' no existe como comando ('type ollama' devuelve
# exito si el programa esta disponible; entonces se evita reimstalarlo).
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi

## Atención

En Colab recuerda arrancar `ollama serve` desde la terminal embebida si todavía no está corriendo. En local basta con tener el servicio levantado de antemano.

## Celda 7 — Terminal embebida (opcional, desactivada)

Hay un paso manual que el notebook no puede hacer por sí solo en Colab: arrancar el **servicio** de Ollama (`ollama serve`), es decir, el proceso que atiende las peticiones del modelo. Para hacerlo a mano, de aquí en adelante aparece una celda desactivada (empieza con `#`) que, al quitarle los `#`, abre una terminal dentro de Colab (`colab-xterm`) donde se escribe `ollama serve &`. Mientras siga comentada no hace nada.

In [5]:
# Descomentar estas dos lineas para abrir una terminal de Colab y arrancar
# a mano el servicio de Ollama con: ollama serve &
#%load_ext colabxterm
#%xterm

Mantendremos `llama3.2:3b` para la demostración.


## Celda 9 — Descarga del modelo

`llama3.2:3b` es un modelo de 3.000 millones de parámetros, pequeño pero capaz, y por eso el elegido para la demo: corre rápido en la GPU gratuita de Colab. `ollama pull` lo descara de la tienda de modelos de Ollama (2.0 GB la primera vez); en ejecuciones posteriores ya no vuelve a descargarlo.

In [19]:
# Descarga el modelo llama3.2:3b (2.0 GB solo la primera vez).
!ollama pull llama3.2:3b

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


## Reutilizamos los mismos servidores MCP

Para que el notebook sea transparente y fácil de depurar, **no** ocultamos la implementación en cadenas largas. Reutilizamos los mismos archivos Python del repositorio para calculadora y clima, igual que en el notebook anterior.

Si el notebook se abre en Colab desde GitHub, ejecuta primero la celda de bootstrap para descargar estos archivos auxiliares en la sesión temporal. En local, el flujo actual se mantiene sin cambios.

## Celda 11 — Localizar los archivos de los servidores MCP

Esta celda no ejecuta nada todavía: solo **busca en disco** los dos archivos de los servidores (calculadora y clima) y guarda sus rutas en las variables `calculator_server` y `weather_server`. Busca en dos carpetas posibles (una para Colab, otra para local) y, si no las encuentra en ninguna, detiene el notebook con un mensaje claro que explica cómo arreglarlo (ejecutar la celda de descarga de la celda 2).

In [ ]:
# 'Path' representa rutas de archivos y carpetas; 'Path.cwd()' es la carpeta
# desde donde corre el notebook.
from pathlib import Path

REPO_ROOT = Path.cwd()
# Dos ubicaciones posibles de los servidores: la del repo del curso (Colab)
# o la carpeta directa (local).
candidate_dirs = [
    REPO_ROOT / 'Sesion6' / 'mcp_servers',
    REPO_ROOT / 'mcp_servers',
]

# 'next((x for x in ...), None)' devuelve el primer elemento que cumple la
# condicion (que la carpeta exista), o None si ninguno la cumple.
SERVERS_DIR = next((path for path in candidate_dirs if path.exists()), None)
if SERVERS_DIR is None:
    msg = (
        'No se encontró la carpeta mcp_servers. '
        'Si estás en Colab, ejecuta primero la celda de bootstrap que descarga los servidores MCP.'
    )
    raise FileNotFoundError(msg)

# Rutas de los dos servidores que este notebook va a usar.
calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
weather_server = SERVERS_DIR / 'weather_mcp_server.py'

# Verificación defensiva: si falta alguno, el error explica qué hacer.
if not calculator_server.exists() or not weather_server.exists():
    msg = (
        f'Faltan archivos MCP en {SERVERS_DIR}. '
        'Si estás en Colab, vuelve a ejecutar la celda de bootstrap.'
    )
    raise FileNotFoundError(msg)

# Un notebook muestra el valor de la última expresión: aquí, la carpeta elegida.
SERVERS_DIR

## Celda 12 — Ver el código del servidor «calculadora»

Imprime el contenido completo del archivo del servidor de la calculadora. El servidor usa la librería `mcp` (clase `FastMCP`), expone **una herramienta** llamada `calculadora` y corre en el puerto HTTP `9080`. Es instructivo leerlo: la herramienta no evalúa el código de la expresión con `eval` (peligroso), sino con un parser restringido de Python que solo acepta las cuatro operaciones y el módulo, de modo que nadie pueda inyectar órdenes maliciosas a través del chat.

In [ ]:
# Muestra el codigo fuente del servidor de la calculadora.
print(f'Servidor calculadora: {calculator_server}')
print('-' * 80)
print(calculator_server.read_text(encoding='utf-8'))

## Celda 13 — Ver el código del servidor «clima»

Imprime el servidor del clima. También expone una sola herramienta (`clima_actual`) en el puerto `9081`; esta consulta una API pública gratuita (Open-Meteo): primero convierte el nombre de la ciudad en coordenadas (geocodificación) y luego pide el clima de esas coordenadas. Son ejemplos canónicos de los dos tipos de herramientas que un LLM necesita: **cálculo local exacto** y **datos externos frescos**.

In [ ]:
# Muestra el codigo fuente del servidor del clima.
print(f'Servidor clima: {weather_server}')
print('-' * 80)
print(weather_server.read_text(encoding='utf-8'))

## Inicio y apagado de servidores MCP

Tienes dos formas válidas de ejecutar estos servidores:

1. Modo automático (recomendado para este notebook):
   Usa la celda de inicio automático para Colab o ejecuta los comandos manualmente.
2. Modo manual (útil para observar procesos):
   Puedes iniciarlos en una terminal aparte y detenerlos con `Ctrl+C`.

Si usas modo manual, **apaga los procesos al terminar** para evitar sesiones colgadas.

Nota: los servidores usan transporte HTTP en los puertos 9080 (calculadora) y 9081 (clima).

## Transporte MCP: stdio vs HTTP

MCP soporta varios mecanismos de transporte para la comunicación entre cliente y servidor:

- **stdio**: Usa flujos estándar (stdin/stdout) para la comunicación. Funciona bien localmente pero tiene problemas en Colab debido a restricciones con descriptores de archivo.
- **HTTP**: Usa un servidor HTTP tradicional. Es más confiable en todos los entornos, especialmente en Colab, y muestra mejor la separación cliente-servidor.

En este notebook usamos **transporte HTTP** con puertos fijos para garantizar compatibilidad tanto local como en Colab, eliminando la necesidad de parches complejos.

- Servidor calculadora: puerto 9080
- Servidor clima: puerto 9081

## Celda 16 — Arranque automático de los servidores (para Colab)

Los servidores son programas independientes: hay que **levantarlos como procesos aparte** antes de que el agente pueda hablar con ellos. En Colab esta celda los arranca en segundo plano con `subprocess.Popen` y espera 2 segundos; en local imprime los comandos para hacerlo a mano en terminales separadas. Usa el transporte HTTP en los puertos 9080 (calculadora) y 9081 (clima).

In [ ]:
# Inicia los servidores MCP en segundo plano (para Colab)
# En local, ejecuta estos comandos en terminales separadas:
#   python Sesion6/mcp_servers/calculator_mcp_server.py
#   python Sesion6/mcp_servers/weather_mcp_server.py

import subprocess
import time

# Iniciar servidores en segundo plano (solo para Colab)
# 'Popen' lanza un programa externo sin bloquear el notebook: el notebook
# sigue ejecutando celdas mientras los dos servidores atienden por HTTP.
# NOTA: 'subprocess.PIPE' en stdout/stderr conecta al notebook el texto que
# los servidores impriman; si el bufer se llena puede bloquear el servidor,
# que es por eso que la solucion propia usa archivos de registro.
if IN_COLAB:
    calculator_proc = subprocess.Popen([
        'python', str(calculator_server)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    weather_proc = subprocess.Popen([
        'python', str(weather_server)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    # Esperar un momento para que los servidores inicien
    time.sleep(2)
    print('Servidores MCP iniciados en segundo plano.')
else:
    print('Ejecutando en entorno local: inicia los servidores manualmente en terminales separadas.')
    print(f'Calculadora: python {calculator_server}')
    print(f'Clima: python {weather_server}')

## Celda 17 — Comandos de arranque manual (ayuda)

No ejecuta servidores: solo **imprime los comandos** que se usarían para arrancarlos a mano en terminales separadas, útil cuando se quiere observar la salida de cada servidor en su propia ventana y poder detenerlos con `Ctrl+C`.

In [ ]:
# 'sys' da acceso a detalles del interprete (aqui se importa pero no se usa).
import sys

# Comandos para iniciar servidores manualmente (en terminales separadas)
# En local: ejecutar en terminales diferentes
# En Colab: usar las celdas de inicio automático o ejecutar con ! en celdas separadas

manual_start_commands = [
    f"python {calculator_server}",
    f"python {weather_server}",
]

print('Comandos para inicio manual:')
for i, cmd in enumerate(manual_start_commands, 1):
    print(f'{i}. {cmd}')

print()
print('Para detener: Ctrl+C en la terminal de cada servidor.')
print('En Colab: usa el menú de procesos para detener servidores en segundo plano.')

## Celda 18 — Arranque automático (repetición de la celda 16)

Esta celda es idéntica a la celda 16: vuelve a arrancar los dos servidores en Colab. Si se ejecuta en secuencia, la segunda pasada lanza procesos adicionales; en la práctica sirve como «vuelta a empezar» por si se reinició la sesión de Colab. En local se limita a reimprimir los comandos manuales.

In [ ]:
# Inicia los servidores MCP en segundo plano (para Colab)
# En local, ejecuta estos comandos en terminales separadas:
#   python Sesion6/mcp_servers/calculator_mcp_server.py
#   python Sesion6/mcp_servers/weather_mcp_server.py

import subprocess
import time

# Iniciar servidores en segundo plano (solo para Colab)
# (Segunda pasada de la celda 16: arrancar de nuevo dos procesos.)
if IN_COLAB:
    calculator_proc = subprocess.Popen([
        'python', str(calculator_server)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    weather_proc = subprocess.Popen([
        'python', str(weather_server)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    # Esperar un momento para que los servidores inicien
    time.sleep(2)
    print('Servidores MCP iniciados en segundo plano.')
else:
    print('Ejecutando en entorno local: inicia los servidores manualmente en terminales separadas.')
    print(f'Calculadora: python {calculator_server}')
    print(f'Clima: python {weather_server}')

## Celda 19 — El cliente MCP, el modelo y el agente (el corazón del notebook)

Es la celda más importante y más larga. En cuatro pasos levanta el sistema completo:

1. **Cliente MCP**: `MultiServerMCPClient` se configura con una entrada por servidor (nombre, transporte `http`, URL con su puerto). `await client.get_tools()` le pregunta a **cada** servidor «¿qué herramientas ofreces?» y las adapta al formato de herramientas de LangChain. Este es el descubrimiento automático que define a MCP: el agente no necesita ninguna línea nueva por herramienta.
2. **Modelo**: `ChatOllama(model='llama3.2:3b', temperature=0)`; la temperatura 0 lo hace determinista, apropiado para un asistente factual.
3. **Agente**: `create_agent(model, tools)` (sobre LangGraph) construye el clásico bucle de agente: el LLM lee la pregunta y la lista de herramientas con sus descripciones, decide si llamar a una y con qué argumentos, recibe el resultado y redacta la respuesta final; puede encadenar varias llamadas.
4. **Funciones de apoyo**: detección de intención por palabras clave (`detect_suggested_tool`), extracción del argumento que exige cada herramienta (`infer_calculator_expression`, `infer_city`), utilidades para aplanar resultados (los mensajes de tool pueden llegar como texto, lista, tupla o diccionario) y un camino de emergencia (`synthesize_from_tool` + `fallback_tool_answer`) que responde con el resultado directo de la herramienta si Ollama no responde.

Sobre los decoradores `await`: en Colab cada celda corre dentro de un bucle de eventos, así que se puede usar `await` directamente sin envolver en `asyncio.run(...)`.

In [7]:
# El modulo 're' es para expresiones regulares (extraer la expresion de un texto).
import re
# 'sys' da acceso a detalles del interprete; aqui solo se usa en el diagnostico.
import sys
# 'create_agent' construye el agente ReAct sobre LangGraph: un LLM que puede
# decidir llamar herramientas y consumir sus resultados antes de responder.
from langchain.agents import create_agent
# El cliente MCP de los adaptadores: abre una sesion con cada servidor.
from langchain_mcp_adapters.client import MultiServerMCPClient
# Conector de LangChain con Ollama (el modelo local).
from langchain_ollama import ChatOllama

# Si una celda previa aplico el parche de compatibilidad de Colab para
# subprocess, se reporta aqui (util para diagnosticar arranques fallidos).
if 'apply_global_subprocess_patch' in globals():
    print(apply_global_subprocess_patch())

# El modelo local que hara el razonamiento; temperature=0 = respuestas
# reproducibles, sin azar creativo, apropiado para un asistente factual.
MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

# --- 1. Cliente MCP: una entrada por servidor ---
# La clave ('calculadora'/'clima') es el nombre con que el agente agrupa las
# tools de ese servidor; el transporte es 'http' y cada servidor tiene su
# puerto fijo (9080 y 9081, ver celdas anteriores).
client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'http',
            'url': 'http://localhost:9080/mcp',  # servidor calculadora
        },
        'clima': {
            'transport': 'http',
            'url': 'http://localhost:9081/mcp',  # servidor clima
        },
    }
)

# --- 2. Descubrimiento automatico de herramientas ---
# 'await' funciona directamente en Colab. get_tools() pregunta a CADA servidor
# por su catalogo y devuelve las tools ya adaptadas al formato de LangChain.
async def load_mcp_tools_with_diagnostics(mcp_client):
    is_colab = bool(globals().get('IN_COLAB', False))
    try:
        return await mcp_client.get_tools()
    except Exception as exc:
        # Diagnostico: en Colab el error 'fileno' indica que el arranque de
        # subprocess stdio fallo por restricciones del entorno.
        if is_colab and 'fileno' in str(exc).lower():
            patched = globals().get('MCP_COLAB_PATCH_APPLIED', False)
            raise RuntimeError(
                'Colab no pudo abrir subprocess stdio (fileno). '
                f'Parche activo={patched}. Ejecuta la celda de compatibilidad Colab y reintenta.'
            ) from exc
        raise

mcp_tools = await load_mcp_tools_with_diagnostics(client)

# --- 3. Agente ReAct con las herramientas descubiertas ---
# No se registran herramientas a mano: el agente las recibe del servidor MCP.
agent_mcp = create_agent(model=llm, tools=mcp_tools)

# Accesorios de consulta sobre el catalogo de tools.
def available_tool_names():
    return [tool.name for tool in mcp_tools]

def tool_from_name(tool_name: str):
    return next((tool for tool in mcp_tools if tool.name == tool_name), None)

# --- Heuristica de intencion del host ---
# Antes de dejar decidir al modelo, el notebook puede 'sugerir' la herramienta
# que la pregunta parece requerir. Es una pista, no una orden: se usa en los
# modos Guiado y Determinista.
def detect_suggested_tool(question: str) -> str | None:
    lower = question.lower()
    weather_signals = ['clima', 'temperatura', 'humedad', 'viento', 'weather', 'lluvia']
    if any(word in lower for word in weather_signals):
        return 'clima_actual'
    has_digit = any(ch.isdigit() for ch in question)
    has_operator = any(op in question for op in ['+', '-', '*', '/', '%', '(', ')'])
    if has_digit and has_operator:
        return 'calculadora'
    return None

def select_tool_name(question: str, preferred_tool: str):
    # Si el usuario eligio una herramienta en la interfaz, manda esa.
    if preferred_tool and preferred_tool != 'Auto':
        return preferred_tool
    return detect_suggested_tool(question)

# --- Extraccion de argumentos para cada herramienta ---
# La herramienta 'calculadora' exige una expresion aritmetica limpia. Esta
# funcion la adivina del texto libre (p. ej. 'cuanto es (125 * 17) + 938').
def infer_calculator_expression(question: str) -> str | None:
    # Extrae una expresion aritmetica probable desde el texto del usuario.
    candidates = re.findall(r"[0-9\s\+\-\*\/%\(\)\.]+", question)
    candidates = [c.strip() for c in candidates if any(ch.isdigit() for ch in c) and any(op in c for op in '+-*/%')]
    if not candidates:
        return None
    return max(candidates, key=len)

# La herramienta del clima exige una ciudad. Heuristica simple: lo que viene
# despues de la palabra 'en', limpio de signos; si no hay nada, Cali.
def infer_city(question: str) -> str:
    # Heuristica simple: toma texto tras 'en ...' y limpia signos frecuentes.
    match = re.search(r"\ben\s+([^\?\.!\n]+)", question, flags=re.IGNORECASE)
    if match:
        city = match.group(1).strip(' ,;')
        if city:
            return city
    return 'Cali, Colombia'

# --- Deteccion del tipo de error de Ollama ---
def is_ollama_connection_error(exc: Exception) -> bool:
    text = str(exc).lower()
    signals = [
        'all connection attempts failed',
        'connection refused',
        'failed to connect',
        'connecterror',
        'could not connect',
    ]
    return any(signal in text for signal in signals)

# --- Aplanado de salidas de herramientas ---
# El resultado de una tool MCP puede llegar como string, lista, tupla o
# diccionario (a veces anidados). Estas funciones lo convierten SIEMPRE en
# texto plano legible por el LLM y por el usuario.
def extract_text_content(content) -> str:
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = [extract_text_content(item) for item in content]
        return '\n'.join(part for part in parts if part)
    if isinstance(content, tuple):
        parts = [extract_text_content(item) for item in content]
        return '\n'.join(part for part in parts if part)
    if isinstance(content, dict):
        if 'text' in content and content['text'] is not None:
            return extract_text_content(content['text'])
        if 'content' in content and content['content'] is not None:
            return extract_text_content(content['content'])
        if 'result' in content and content['result'] is not None:
            return extract_text_content(content['result'])
        return str(content)
    text_attr = getattr(content, 'text', None)
    if text_attr is not None:
        return extract_text_content(text_attr)
    content_attr = getattr(content, 'content', None)
    if content_attr is not None:
        return extract_text_content(content_attr)
    result_attr = getattr(content, 'result', None)
    if result_attr is not None:
        return extract_text_content(result_attr)
    return str(content)

def normalize_tool_output(tool_output) -> str:
    return extract_text_content(tool_output)

# --- Camino de emergencia si el LLM no responde ---
# Si la herramienta funciono pero Ollama esta caido, se devuelve el resultado
# directo de la herramienta con una nota, en lugar de colapsar la demo.
def fallback_tool_answer(tool_name: str, tool_output, llm_error: str | None = None) -> str:
    text = normalize_tool_output(tool_output)
    if llm_error:
        return (
            f"Resultado directo de la herramienta {tool_name}:\n{text}\n\n"
            f"Nota: no pude contactar Ollama para redactar una respuesta ({llm_error})."
        )
    return f"Resultado directo de la herramienta {tool_name}:\n{text}"

async def run_selected_tool(tool_name: str, question: str):
    tool = tool_from_name(tool_name)
    if tool is None:
        raise ValueError(f'Herramienta no encontrada: {tool_name}')

    # Cada herramienta espera argumentos con nombre; se infieren del texto.
    if tool_name == 'calculadora':
        expression = infer_calculator_expression(question)
        if not expression:
            raise ValueError(
                'No pude inferir una expresion aritmetica del mensaje. '
                'Escribe algo como: "Cuanto es (125 * 17) + 938?".'
            )
        payload = {'expression': expression}
    elif tool_name == 'clima_actual':
        payload = {'city': infer_city(question)}
    else:
        raise ValueError(f'Herramienta no soportada: {tool_name}')

    return await tool.ainvoke(payload)

def tool_message_used(messages) -> bool:
    return any(getattr(msg, 'type', '') == 'tool' for msg in messages)

# --- Redaccion final a partir del resultado de una herramienta ---
# Modo 'determinista': la tool corre y el LLM SOLO redacta la respuesta con
# ese dato (no puede inventar numeros). Si Ollama cae, el fallback responde
# con el dato crudo de la herramienta.
async def synthesize_from_tool(question: str, tool_name: str, tool_output):
    try:
        normalized_tool_output = normalize_tool_output(tool_output)
        synthesis = llm.invoke([
            ('system', 'Responde en espanol, de forma breve y usando solamente el resultado de la herramienta.'),
            ('user', f"Pregunta: {question}\n\nSalida de {tool_name}: {normalized_tool_output}\n\nRedacta una respuesta final clara."),
        ])
        return extract_text_content(synthesis.content)
    except Exception as exc:
        if is_ollama_connection_error(exc):
            return fallback_tool_answer(tool_name, tool_output, llm_error=str(exc))
        raise

## Del agente a la interfaz conversacional

Aquí reaparece una idea de la sesión de RAG: Gradio maneja historial, mientras reconstruimos mensajes para el agente.

En esta versión añadimos controles de estrategia para clase:

- Autónomo: el agente decide libremente.
- Guiado: sugerimos tool preferida y permitimos fallback opcional.
- Determinístico: ejecutamos una tool explícita y luego redactamos la respuesta final.

Nota de diseño de UX: el único input principal es el chat. No pedimos campos separados para cálculo o clima porque la intención idealmente se infiere desde el mensaje del usuario. El selector de tool funciona como preferencia/guía, no como un formulario obligatorio.

## Celda 22 — La función `respond`: un turno del chatbot en los tres modos

Esta celda define toda la maquinaria del turno de conversación:

- `history_to_messages`: Gradio guarda el historial como lista de diccionarios (`{'role': ..., 'content': ...}`); esta función lo convierte a la lista de mensajes `(rol, texto)` que alimenta al agente, y añade al final la nueva pregunta.
- `compact_trace_text`: arma una línea corta con la traza de lo ocurrido (modo, herramienta usada, si hubo fallback) para que la interfaz pueda mostrarla.
- `recover_with_direct_tool_if_possible`: rescate si el agente no puede hablar con Ollama: ejecuta la herramienta directamente y devuelve su resultado.
- `respond`: la función principal que la interfaz llamará en cada mensaje. Según el modo:
  - **Deterministic**: elige la herramienta por la heurística y la ejecuta directamente; el LLM solo redacta la respuesta final con el dato. Si no hay señal de herramienta, informa.
  - **Guided** (por defecto): llama al agente con un mensaje de sistema que prioriza la herramienta sugerida. Si el agente terminó sin usarla y el fallback está activado, ejecuta la herramienta a mano y redacta con ella (la traza lo marca con `guided_fallback_used=True`).
  - **Autonomous**: simplemente entrega la pregunta al agente, sin guía alguna.
- Si durante el turno Ollama no responde (conexión), se intenta el rescate; si no se puede, se le explica al usuario qué hacer.

Al final, cada turno registra la pregunta y la respuesta en el historial de Gradio (el valor devuelto `''` limpia el cuadro de texto).

In [8]:
import gradio as gr

def history_to_messages(question, chat_history):
    # Convierte el historial de Gradio en la lista de mensajes del agente.
    messages = []
    for item in (chat_history or []):
        role = item.get('role')
        content = item.get('content')
        if role in {'user', 'assistant'} and content is not None:
            messages.append((role, normalize_content(content)))
    messages.append(('user', question))
    return messages

def normalize_content(content):
    return extract_text_content(content)

def compact_trace_text(trace: dict):
    # Linea resumida de la traza MCP, para mostrarla bajo la respuesta.
    lines = [
        f"modo={trace.get('mode')}",
        f"used_tool={trace.get('used_tool')}",
        f"tool_selected={trace.get('tool_selected')}",
        f"guided_fallback_used={trace.get('guided_fallback_used')}",
    ]
    if trace.get('tool_error'):
        lines.append(f"tool_error={trace.get('tool_error')}")
    return ' | '.join(lines)

async def recover_with_direct_tool_if_possible(question: str, selected_tool: str | None, trace: dict, llm_exc: Exception):
    # Rescate: si Ollama esta caido pero la herramienta vive, se devuelve el
    # resultado directo de la herramienta (la demo no muere).
    if selected_tool is None:
        return None
    try:
        tool_output = await run_selected_tool(selected_tool, question)
        trace['used_tool'] = True
        trace['guided_fallback_used'] = True
        return fallback_tool_answer(selected_tool, tool_output, llm_error=str(llm_exc))
    except Exception:
        return None

async def respond(
    question,
    chat_history,
    mode,
    preferred_tool,
    guided_fallback,
    show_trace,
 ):
    # 'respond' es la funcion que la interfaz llamara en cada mensaje.
    chat_history = chat_history or []
    mode = mode or 'Guided'
    messages = history_to_messages(question, chat_history)
    selected_tool = select_tool_name(question, preferred_tool)
    trace = {
        'mode': mode,
        'tool_selected': selected_tool,
        'used_tool': False,
        'guided_fallback_used': False,
        'tool_error': None,
    }

    try:
        if mode == 'Deterministic':
            # Modo determinista: la tool corre directo y el LLM solo redacta.
            if selected_tool is None:
                answer = (
                    'No pude inferir una herramienta automáticamente. '
                    'Selecciona una herramienta o ajusta tu pregunta.'
                )
            else:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True

        elif mode == 'Guided':
            # Modo guiado: se antepone un mensaje de sistema que prioriza la
            # herramienta sugerida, pero el agente sigue decidiendo.
            guided_messages = messages
            if selected_tool is not None:
                guided_messages = [
                    (
                        'system',
                        f"Si la pregunta requiere cálculo o datos externos, prioriza usar la herramienta {selected_tool} antes de responder.",
                    )
                ] + guided_messages

            result = await agent_mcp.ainvoke({'messages': guided_messages})
            used_tool = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)
            trace['used_tool'] = used_tool

            # Fallback opt-in: si el agente termino SIN usar la herramienta y
            # el usuario activo el fallback, se ejecuta a mano y se redacta.
            if guided_fallback and (not used_tool) and selected_tool is not None:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True
                trace['guided_fallback_used'] = True

        else:
            # Modo autonomo: el agente decide libremente, sin pistas del host.
            result = await agent_mcp.ainvoke({'messages': messages})
            trace['used_tool'] = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)

    except Exception as exc:
        trace['tool_error'] = str(exc)
        if is_ollama_connection_error(exc):
            # Si Ollama no responde, se intenta el rescate con la herramienta.
            recovered = await recover_with_direct_tool_if_possible(question, selected_tool, trace, exc)
            if recovered is not None:
                answer = recovered
            else:
                answer = (
                    'No pude conectar con Ollama para generar respuesta del agente. '
                    'Inicia Ollama con `ollama serve` o haz una pregunta que active calculadora/clima.'
                )
        else:
            answer = f"Ocurrió un error: {exc}"

    answer = normalize_content(answer)

    if show_trace:
        answer = f"{answer}\n\n---\nTraza MCP: {compact_trace_text(trace)}"

    # Se anexan el turno (pregunta y respuesta) al historial de Gradio y se
    # devuelve '' para vaciar el cuadro de texto.
    updated_history = chat_history + [
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]
    return '', updated_history

def reset_chat():
    # Boton Limpiar: historial vacio y cuadro de texto vacio.
    return '', []

## Lanzando la interfaz de chat

Esta versión es opcional precisamente porque ya no enseña un concepto nuevo de NLP o de MCP; enseña cómo empaquetar el agente en una demo usable. Es útil para estudiantes que quieran presentar el flujo completo de extremo a extremo.

## Celda 24 — Construcción y lanzamiento de la interfaz

`gr.Blocks` es el lienzo de Gradio: en él se apilan los controles de arriba abajo. Hay dos desplegables (modo de ejecución y herramienta preferida), dos casillas (fallback guiado y ver la traza MCP), el chat y una casilla de texto. La parte clave es el **cableado**: `msg.submit(...)` conecta el evento «el usuario envía un mensaje» con la función `respond`, diciendo cuál control alimenta cada argumento de la función (por eso el orden de esa lista debe coincidir con los parámetros de `respond`) y qué controles se actualizan con el resultado (`[msg, chatbot]`: el cuadro de texto queda vacío y el historial crece). `launch(inline=False)` levanta el servidor web de Gradio y muestra la URL; para la demo del taller, el video comienza precisamente ejecutando esta celda.

Gradio imprime la URL local (`http://127.0.0.1:7860`); en Colab aparece un enlace temporal `*.gradio.live`, que cambia cada vez que se lanza (lección aprendida en el taller anterior: esos enlaces mueren con cada desconexión).

In [9]:
# 'gr.Blocks' abre el lienzo de la interfaz; todo lo que este dentro del
# 'with' se dibuja de arriba hacia abajo.
with gr.Blocks() as gr_blocks:
    gr.Markdown('## Chat con herramientas MCP')

    # Desplegable del modo de ejecucion (Guided por defecto).
    mode = gr.Dropdown(
        choices=['Guided', 'Autonomous', 'Deterministic'],
        value='Guided',
        label='Modo de ejecución',
    )
    # Desplegable de tool preferida: 'Auto' delega en la heuristica.
    preferred_tool = gr.Dropdown(
        choices=['Auto', 'calculadora', 'clima_actual'],
        value='Auto',
        label='Tool preferida',
    )
    # En modo Guided, si el agente no uso la herramienta sugerida, ejecutarla
    # a mano y redactar con su salida (casilla activada por defecto).
    guided_fallback = gr.Checkbox(
        label='En modo Guided, usar fallback de tool si el agente no la invoca',
        value=True,
    )
    # Mostrar la linea de traza MCP bajo cada respuesta (desactivada).
    show_trace = gr.Checkbox(
        label='Mostrar traza de uso MCP',
        value=False,
    )

    # El chat: historial de mensajes y cuadro para el nuevo mensaje.
    chatbot = gr.Chatbot(label='Historial')
    msg = gr.Textbox(
        label='¿Qué quieres preguntar?',
        placeholder='Ejemplo: ¿Cuál es la temperatura actual en Cali? o ¿Cuánto es (125 * 17) + 938?',
    )
    clear = gr.Button('Limpiar')

    # Cableado del evento 'usuario envia mensaje': se pasan, en este orden,
    # msg, chatbot, mode, preferred_tool, guided_fallback, show_trace como
    # argumentos de respond (mismo orden que sus parametros); la salida
    # actualiza [msg, chatbot].
    msg.submit(
        respond,
        [
            msg,
            chatbot,
            mode,
            preferred_tool,
            guided_fallback,
            show_trace,
        ],
        [msg, chatbot],
    )
    # El boton Limpiar reinicia la UI sin pasar por el agente (queue=False
    # lo ejecuta sin esperar turnos pendientes).
    clear.click(reset_chat, None, [msg, chatbot], queue=False)

# Levanta el servidor web de Gradio y imprime la URL para abrir el chat.
gr_blocks.launch(inline=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Celda 25 — Cierre de la interfaz

Detiene el servidor web de Gradio y libera el puerto 7860. Conviene ejecutarlo al terminar la demo (o antes de relanzar la interfaz), porque en Colab un servidor abierto retiene recursos y, al desconectar la sesión, el enlace `*.gradio.live` muere igualmente.

In [10]:
# Detiene el servidor de Gradio y libera el puerto 7860.
gr_blocks.close()

Closing server running on port: 7860


## Conclusiones

- Este notebook solo agrega una capa de presentación; el corazón sigue siendo el mismo agente con herramientas MCP.
- Gradio permite convertir el experimento técnico en una demo conversacional muy rápidamente.
- Como material opcional, ayuda a cerrar la sesión con una visión más aplicada sin sobrecargar el flujo conceptual principal.
